# Project 3 · Build a Mini GPT
**AI Workshop – From MLP to Transformers**

From images to text: a small Transformer that writes Shakespeare-like text, one character at a time. We will:
1. **Tokenize the text** into characters and ids.
2. **Build the Transformer**: embeddings, masked self-attention, feed-forward MLP.
3. **Train**, then **load a checkpoint**.
4. **Generate text** and play with the temperature.

> **Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom (**Shift + Enter**).

## 0 · Setup

In [ ]:
import math
import os
import time
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

## 01 · Tokenize the text
Tiny Shakespeare: about 1 MB of plays. Our tokens are single **characters**, so the vocabulary stays tiny.

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("shakespeare.txt"):
    urllib.request.urlretrieve(url, "shakespeare.txt")
text = open("shakespeare.txt", encoding="utf-8").read()

print(f"{len(text):,} characters")
print(text[:400])

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}   # character → id
itos = {i: ch for ch, i in stoi.items()}       # id → character

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print("Vocabulary size:", vocab_size)
print("".join(chars))
print('encode("hello") →', encode("hello"))
print('decode(...)     →', decode(encode("hello")))

Turn the whole text into one long row of ids, and keep the last 10% aside to check the model on text it hasn't trained on.

Training examples are random slices of **128 characters**. The target is the same slice **shifted by one**: at every position, the model must predict the next character.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 128   # context length: how many characters the model sees at once
batch_size = 64

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

x, y = get_batch("train")
print("input :", repr(decode(x[0, :40].tolist())))
print("target:", repr(decode(y[0, :40].tolist())))

## 02 · Build the Transformer
The pieces from the slides:
- **Token + position embeddings**: each id becomes a learned vector, plus a vector for where it sits.
- **Masked self-attention**: `softmax(QKᵀ / √d) · V`, with a causal mask so a token can only look back.
- **Feed-forward MLP**: Project 1 is back, applied to every token on its own.
- **Residual connections and LayerNorm** keep the stack trainable (the GPT "pre-norm" block).

In [ ]:
n_embd = 192     # size of each token vector
n_head = 6       # attention heads running in parallel
n_layer = 4      # Transformer blocks stacked
dropout = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(n_embd, 3 * n_embd)   # makes queries, keys and values at once
        self.proj = nn.Linear(n_embd, n_embd)
        self.drop = nn.Dropout(dropout)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.keep_attention = False                # set True to store the weights for plotting
        self.attention = None

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(n_embd, dim=2)
        # split into heads: B × heads × T × head_size
        q, k, v = [t.view(B, T, n_head, C // n_head).transpose(1, 2) for t in (q, k, v)]
        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))     # every query · every key
        att = att.masked_fill(self.mask[:T, :T] == 0, float("-inf"))  # causal mask: hide the future
        att = F.softmax(att, dim=-1)                                  # each row sums to 1
        if self.keep_attention:
            self.attention = att.detach().cpu()
        y = self.drop(att) @ v                                        # weighted mix of the values
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(y))

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(n_embd), CausalSelfAttention()
        self.ln2, self.ffwd = nn.LayerNorm(n_embd), FeedForward()

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # tokens exchange information (+ residual)
        x = x + self.ffwd(self.ln2(x))   # each token is processed on its own (+ residual)
        return x

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)   # a score for every possible next character

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        logits = self.head(self.ln_f(self.blocks(x)))
        loss = None
        if targets is not None:   # cross-entropy, exactly as in Project 1
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

model = MiniGPT().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 03 · Train, then load a checkpoint
Same recipe as every project: forward → loss → backward → update.
A random guess among 65 characters gives a loss of about ln 65 ≈ **4.17**; watch it fall.

In [ ]:
@torch.no_grad()
def estimate_loss(iters=50):
    model.eval()
    out = {}
    for split in ("train", "val"):
        out[split] = sum(model(*get_batch(split))[1].item() for _ in range(iters)) / iters
    model.train()
    return out

MAX_STEPS = 3000      # about 2–3 minutes on a T4 GPU
EVAL_EVERY = 250
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
history = []

model.train()
start = time.time()
for step in range(MAX_STEPS + 1):
    if step % EVAL_EVERY == 0:
        losses = estimate_loss()
        history.append((step, losses["train"], losses["val"]))
        print(f"step {step:5d} · train loss {losses['train']:.3f} · val loss {losses['val']:.3f} · {time.time() - start:.0f}s")
    if step == MAX_STEPS:
        break
    x, y = get_batch("train")
    _, loss = model(x, y)              # forward + loss
    optimizer.zero_grad()
    loss.backward()                    # backpropagation
    optimizer.step()                   # update the weights

In [ ]:
steps, train_l, val_l = zip(*history)
plt.figure(figsize=(7, 3.5))
plt.plot(steps, train_l, label="train"); plt.plot(steps, val_l, label="validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("Mini GPT training")
plt.show()

torch.save({"model": model.state_dict(), "chars": chars}, "mini_gpt.pt")
print("Saved mini_gpt.pt")

### Load a checkpoint
A model trained for longer writes much better text. If you have a checkpoint trained with the **same settings**
(for example, the instructor's, made by running this notebook with `MAX_STEPS = 10000`), drag it into the
**Files** panel on the left, name it `mini_gpt_trained.pt`, and run the cell below.
Without it, we simply keep the model we just trained.

In [ ]:
CHECKPOINT = "mini_gpt_trained.pt"

if os.path.exists(CHECKPOINT):
    ckpt = torch.load(CHECKPOINT, map_location=device)
    assert ckpt["chars"] == chars, "This checkpoint was trained on a different text."
    model.load_state_dict(ckpt["model"])
    print("Loaded", CHECKPOINT)
else:
    print(f"No {CHECKPOINT} found: keeping the model we just trained.")

## 04 · Generate text
Generation is next-token prediction on repeat: **sample a character, append it, feed the sequence back in**.

> ✏️ **Your turn:** replace every `TODO` in the cell below, then run it. Stuck? The solution is in the **Solutions** section at the end of the notebook.

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0):
    model.eval()
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])           # never more than the context length
        # ✏️ Fill in: predict → sample → append, then repeat
        logits = logits[:, -1, :] / TODO                    # scores for the NEXT character, divided by the temperature
        probs = TODO                                        # scores → probabilities (softmax over the last dim)
        next_id = TODO                                      # sample one token from probs (torch.multinomial)
        idx = TODO                                          # append next_id to idx (torch.cat along dim=1)
    return idx

Now compare an untrained model with ours.

In [ ]:
def write(prompt="\n", length=300, temperature=1.0, gpt=None):
    gpt = gpt or model
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    return decode(generate(gpt, idx, length, temperature)[0].tolist())

print("──── untrained model ────")
print(write("ROMEO:\n", length=200, gpt=MiniGPT().to(device)))
print()
print("──── our trained model ────")
print(write("ROMEO:\n", length=500, temperature=0.8))

### Next-token probabilities
What does the model think comes next? (Compare with the "the ca" slide.)

In [ ]:
def next_char_probs(context, top=5):
    model.eval()
    idx = torch.tensor([encode(context)], dtype=torch.long, device=device)
    with torch.no_grad():
        probs = F.softmax(model(idx)[0][0, -1], dim=-1).cpu()
    p, i = probs.topk(top)
    for prob, ch in zip(p.tolist(), i.tolist()):
        print(f"{itos[ch]!r:6} {prob:.2f}  " + "█" * int(prob * 40))

next_char_probs("the ca")

### Temperature
Low temperature: safe and repetitive. High temperature: creative and noisy.

In [ ]:
for t in (0.3, 1.0, 1.5):
    print(f"──── temperature {t} ────")
    print(write("JULIET:\n", length=250, temperature=t))
    print()

### Look inside: attention weights
Each row is a character asking; each column is a character it looks at. The upper triangle is empty: that's the causal mask.

In [ ]:
sentence = "the cat sat on the mat"
attn = model.blocks[0].attn
attn.keep_attention = True
model.eval()
with torch.no_grad():
    model(torch.tensor([encode(sentence)], device=device))
attn.keep_attention = False

weights = attn.attention[0, 0]          # first block, first head
plt.figure(figsize=(6.5, 6))
plt.imshow(weights, cmap="Blues")
plt.xticks(range(len(sentence)), list(sentence))
plt.yticks(range(len(sentence)), list(sentence))
plt.title("Block 1 · head 1")
plt.show()

### Try it yourself
- Write your own prompt in `write("...")`.
- Make the model bigger (`n_layer = 6`, `n_embd = 384`) and train longer: does the validation loss go lower?
- Look at other heads: `attn.attention[0, h]` for h = 0…5, or other blocks.

## 🔑 Solutions
Only if you are stuck: run the solution for your step (double-click to see the code), then go back and continue where you were.

In [ ]:
# @title Solution · generate
@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0):
    model.eval()
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])           # never more than the context length
        logits = logits[:, -1, :] / temperature            # scores for the NEXT character only
        probs = F.softmax(logits, dim=-1)                  # scores → probabilities
        next_id = torch.multinomial(probs, num_samples=1)  # sample one token…
        idx = torch.cat([idx, next_id], dim=1)             # …append it, and repeat
    return idx